<div style="position: relative; background: linear-gradient(135deg, #24398A 0%, #1a2a66 100%); border-radius: 20px 20px 0px 0px; padding: 30px; box-shadow: 0 8px 16px rgba(36, 57, 138, 0.3);">
  
  <!-- Logo UNISON - Derecha -->
  <div style="position: absolute; top: 20px; right: 20px; background: white; padding: 4px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">
    <img src="images/logo_unison.jpg" alt="UNISON" style="height: 160px;">
  </div>
  
  <!-- Imagen Izquierda -->
  <div style="position: absolute; top: 20px; left: 20px; background: white; padding: 4px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">
    <img src="images/es_banner.jpg" alt="Curso" style="height: 160px;">
  </div>
   
  <!-- Título centrado -->
  <div style="color: white; max-width: 60%; margin: 0 auto; text-align: center;">
    <h1 style="color: #EBA93B; margin: 0; font-size: 28px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
      Metaheurísticas para Ciencia de Datos: <br>Teoría y Práctica
    </h1><br>
    <h3 style="margin: 15px 0 5px 0; font-size: 16px; opacity: 0.95;">Maestría en Ciencia de Datos</br></br>
    Ramón Soto C. / ramon.soto@unison.mx</h3>
  </div>
</div>

<div style="background: white; border: 3px solid #EBA93B; border-radius: 0px 0px 20px 20px; padding: 25px;">
  <div style="display: flex; align-items: center; margin-bottom: 15px;">
    <div style="background: #24398A; color: white; font-size: 24px; font-weight: bold; padding: 10px 20px; border-radius: 8px; margin-right: 20px;">04</div>
    <div>
      <h2 style="color: #24398A; margin: 0;">Estrategias Evolutivas: Cuando el algoritmo aprende a mutar</h2>
      <p style="color: #666; margin: 5px 0 0 0; font-style: italic;">De la representación binaria a la autoadaptación en espacios continuos</p>
    </div>
  </div>
  
<div style="background: #f8f9fa; padding: 15px; border-radius: 5px; border-left: 4px solid #24398A;">
    <h4 style="color: #24398A; margin-top: 0;"><strong>Presentación general</strong></h4>
    <p style="line-height: 1.7; color: #333;">
    En la lección anterior construimos el Algoritmo Genético Simple (AGS): una herramienta versátil para explorar paisajes multimodales. Vimos que funciona notablemente bien en Schwefel y en funciones multimodales bidimensionales. Pero hay una limitación que pasamos por alto: el AGS no sabe cuánto mutar. La tasa de mutación σ es fija, elegida por el diseñador antes de correr el algoritmo. Si elegimos σ demasiado grande, el algoritmo explora sin converger; si lo elegimos demasiado pequeño, converge prematuramente en un óptimo local.
    </p>
    <p style="line-height: 1.7; color: #333;">
    Las <strong>Estrategias Evolutivas (EE)</strong> resuelven este problema de raíz: hacen que σ sea parte del cromosoma y lo dejan evolucionar junto con las variables del problema. El resultado es un algoritmo que <em>aprende a mutar mientras optimiza</em>—una forma de meta-optimización que lo convierte en la variante más poderosa de la computación evolutiva para problemas continuos de ingeniería y ciencia de datos.
    </p>
    <p style="line-height: 1.7; color: #333;">
    Esta lección hace el recorrido completo: desde la comparativa estructural con el AGS, pasando por la geometría de la mutación gaussiana y la autoadaptación lognormal, hasta CMA-ES —el estado del arte en optimización sin gradiente para espacios continuos.
    </p>
  </div>
</div>

<div style="padding: 15px 20px; background-color: #f8f9fa; margin: 20px 0;">
    <p style="font-size: 16px; line-height: 1.8; color: #2c3e50; margin: 0 0 15px 0;">
        En la lección anterior construimos el AGS desde sus cimientos: representación binaria, selección por torneo, cruza de un punto, mutación por flip de bits. Lo aplicamos a Schwefel y a una función multimodal bidimensional y vimos que, pese a su simplicidad, logra aproximarse al óptimo global. También vimos su limitación central: los parámetros de variación son fijos. El algoritmo no aprende de su propio comportamiento.
    </p>
    <p style="font-size: 16px; line-height: 1.8; color: #2c3e50; margin: 0;">
        Las <strong>Estrategias Evolutivas</strong> nacieron en los laboratorios de ingeniería aeronáutica alemana en los años 60 con una pregunta distinta: ¿qué pasa si los parámetros que controlan la variación también evolucionan? La respuesta a esa pregunta es una familia de algoritmos que representa el estado del arte en optimización continua sin gradiente.
    </p>
</div>

In [ ]:
# CONFIGURACIÓN INICIAL
# =====================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap

# Configuración estética
np.set_printoptions(precision=4, suppress=True)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

# Reproducibilidad
rng = np.random.default_rng(42)

print("Librerías cargadas exitosamente.\n")

## 1. Del AGS a las EE: ¿qué cambia?

El AGS y las Estrategias Evolutivas comparten la misma estructura de alto nivel: inicializar una población, evaluar aptitud, seleccionar, variar, repetir. Pero difieren en casi todas las decisiones de diseño concretas. La siguiente tabla sitúa exactamente dónde.

| Aspecto | AGS (lección anterior) | Estrategias Evolutivas |
|---|---|---|
| **Representación** | Cadena binaria (genotipo → fenotipo) | Vector real de variables + vector de σ |
| **Espacio natural** | Discreto / combinatorio | Continuo, espacios de alta dimensión |
| **Variación principal** | Cruza de un punto + flip de bits | Mutación gaussiana $x' = x + \mathcal{N}(0, \sigma^2 I)$ |
| **Parámetros de variación** | Fijos (diseñados a priori) | **Evolucionan junto con las variables** |
| **Selección** | Torneo probabilístico | Determinista: los μ mejores de λ hijos |
| **Elitismo** | Opcional | Estructural: notación **(μ+λ)** o **(μ,λ)** |
| **Garantías teóricas** | Esquema de Holland (bloques de construcción) | Regla de 1/5, convergencia en esfera |
| **Caso de uso típico** | Problemas discretos, combinatorios, binarios | Optimización continua, ingeniería, calibración |

<br>

La diferencia clave está en la última fila de la representación: el cromosoma de una EE tiene **dos capas**:

$$\text{individuo} = (\underbrace{x_1, x_2, \ldots, x_n}_{\text{variables del problema}}, \underbrace{\sigma_1, \sigma_2, \ldots, \sigma_n}_{\text{parámetros de estrategia}})$$

Los $x_i$ son la solución. Los $\sigma_i$ son instrucciones sobre *cómo mutar* esa solución. Ambos evolucionan simultáneamente.

<div style="background: linear-gradient(135deg, #e8f5e9, #c8e6c9); border-left: 5px solid #2e7d32; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>💡 Intuición central:</strong> En el AGS fijamos σ antes de correr el algoritmo, igual que un cocinero que decide de antemano cuánta sal usar en todos los platos. Las EE dejan que σ evolucione: el algoritmo prueba distintas intensidades de mutación y conserva las que producen mejores resultados. El algoritmo aprende a sazonar mientras cocina.
</div>

### 1.1 El problema de σ fija: una demostración

Antes de ver cómo las EE resuelven esto, conviene ver exactamente qué le pasa a un optimizador con σ fija cuando el paisaje tiene zonas de distinta rugosidad. La siguiente celda lo ilustra sobre la función de Rosenbrock en 2D:

$$f(x, y) = (1 - x)^2 + 100(y - x^2)^2$$

Esta función tiene un valle curvo y estrecho que lleva al óptimo en $(1, 1)$. En ese valle, un σ grande salta por encima del fondo; un σ pequeño avanza demasiado lento en la entrada del valle.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PROBLEMA DE σ FIJA EN ROSENBROCK
# Mostramos cómo distintos valores de σ fija generan comportamientos
# cualitativamente diferentes sobre el mismo paisaje.
# ════════════════════════════════════════════════════════════════════════

def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100*(y - x**2)**2

def ags_sigma_fija(sigma, n_gen=80, pop_size=30, seed=0):
    """AGS minimalista con σ fija sobre Rosenbrock 2D."""
    rng_local = np.random.default_rng(seed)
    pop = rng_local.uniform(-2, 2, (pop_size, 2))
    best_fitness = []
    best_ind = []

    for _ in range(n_gen):
        fitness = np.array([-rosenbrock(ind) for ind in pop])
        best_idx = np.argmax(fitness)
        best_fitness.append(-fitness[best_idx])
        best_ind.append(pop[best_idx].copy())

        # Selección por torneo (k=3)
        new_pop = []
        for _ in range(pop_size):
            idx = rng_local.choice(pop_size, 3, replace=False)
            winner = pop[idx[np.argmax(fitness[idx])]]
            child = winner + rng_local.normal(0, sigma, 2)
            child = np.clip(child, -2, 2)
            new_pop.append(child)
        pop = np.array(new_pop)

    return best_fitness, best_ind


# Ejecutar con 3 valores de sigma
sigmas = [0.05, 0.3, 1.0]
colores = ['#2196F3', '#4CAF50', '#F44336']
etiquetas = ['σ = 0.05 (demasiado pequeña)', 'σ = 0.3 (moderada)', 'σ = 1.0 (demasiado grande)']
resultados = {s: ags_sigma_fija(s) for s in sigmas}

# ── Figura ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Impacto de σ fija en el AGS sobre Rosenbrock 2D', fontsize=14, fontweight='bold')

# Paisaje Rosenbrock
xx, yy = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-2, 2, 300))
zz = (1 - xx)**2 + 100*(yy - xx**2)**2
zz_log = np.log1p(zz)  # escala logarítmica para visualizar

for ax, sigma, color, etiqueta in zip(axes, sigmas, colores, etiquetas):
    bf, traj = resultados[sigma]
    traj = np.array(traj)

    ax.contourf(xx, yy, zz_log, levels=30, cmap='YlOrRd', alpha=0.7)
    ax.contour(xx, yy, zz_log, levels=10, colors='white', alpha=0.3, linewidths=0.5)
    ax.plot(traj[:, 0], traj[:, 1], '-o', color=color, markersize=3,
            linewidth=1.5, label='Trayectoria del mejor', alpha=0.8)
    ax.plot(1, 1, '*', color='gold', markersize=15, zorder=5, label='Óptimo (1,1)')
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_title(etiqueta, fontsize=11)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.legend(fontsize=8, loc='upper left')

    # Indicar valor final
    ax.annotate(f'f* = {bf[-1]:.2f}', xy=(0.98, 0.04),
                xycoords='axes fraction', ha='right', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

plt.tight_layout()
plt.show()

# ── Curvas de convergencia ─────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4))
for sigma, color, etiqueta in zip(sigmas, colores, etiquetas):
    bf, _ = resultados[sigma]
    ax2.semilogy(bf, color=color, linewidth=2, label=etiqueta)

ax2.axhline(0, color='gold', linestyle='--', linewidth=1.5, label='Óptimo global (f=0)')
ax2.set_xlabel('Generación'); ax2.set_ylabel('Mejor f(x) — escala log')
ax2.set_title('Curvas de convergencia: σ fija en Rosenbrock', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Qué observar en estas gráficas

Las trayectorias sobre el mapa de contorno revelan los tres comportamientos patológicos del σ fijo:

Con **σ pequeña**, la trayectoria avanza milímetro a milímetro. Al principio parece prometedora, pero se detiene mucho antes de alcanzar el óptimo: queda atrapada en la curvatura del valle porque sus pasos son demasiado cortos para seguir el fondo.

Con **σ grande**, la trayectoria cruza el valle varias veces pero nunca baja a su fondo. Las mutaciones son tan intensas que el algoritmo salta por encima de la región de alta calidad sin poder asentarse en ella.

Con **σ moderada**, el resultado mejora pero tampoco es óptimo: σ = 0.3 funciona bien en la fase de exploración inicial pero es demasiado grande para el afinamiento final.

La curva de convergencia lo confirma en escala logarítmica: ningún valor fijo de σ es bueno en todas las fases del proceso. Lo que se necesita es un σ que comience grande y se vaya reduciendo a medida que el algoritmo converge — exactamente lo que ofrecen las EE.

<div style="background: linear-gradient(135deg, #fff3e0, #ffe0b2); border-left: 5px solid #e65100; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>⚠️ El dilema del parámetro fijo:</strong> Elegir σ "a mano" requiere conocer de antemano la geometría del paisaje que se quiere explorar. Pero si ya conociéramos esa geometría, no necesitaríamos el algoritmo. La autoadaptación es la salida lógica a esta circularidad.
</div>

### 1.2 Historia y motivación original

Las Estrategias Evolutivas nacieron entre 1963 y 1965 en la Technische Universität Berlin, de la mano de Ingo Rechenberg y Hans-Paul Schwefel, como parte de sus tesis doctorales de ingeniería aeronáutica. El contexto es revelador: querían optimizar la forma de una placa articulada sumergida en un flujo de agua. No existía gradiente analítico. No había buenas aproximaciones lineales. El espacio de diseño era continuo y de dimensión moderada.

Su solución fue ingeniosamente simple: representar el diseño como un vector real, perturbarlo con ruido gaussiano, y quedarse con la perturbación si mejoraba. De ahí nació la **(1+1)-ES** — un padre, un hijo, selección del mejor.

El salto conceptual clave vino después: Rechenberg observó empíricamente que la tasa de éxito óptima de las mutaciones es aproximadamente **1/5**. Si más de un quinto de las mutaciones mejoran la solución, σ es demasiado pequeña (poca exploración). Si menos de un quinto mejoran, σ es demasiado grande (demasiado ruido). Esta observación —la **Regla de los 1/5**— fue el primer mecanismo de adaptación de parámetros de estrategia.

```
  1963-65   Rechenberg & Schwefel: (1+1)-ES, optimización de forma
    │
  1973      Tesis doctoral Rechenberg: Regla de los 1/5
    │
  1975      Schwefel: (μ+λ)-ES y (μ,λ)-ES con poblaciones
    │
  1987      Bäck & Schwefel: autoadaptación lognormal de σ
    │
  1996      Hansen & Ostermeier: CMA-ES — el estado del arte actual
```

Esta genealogía importa porque cada paso resolvió una limitación concreta del anterior, y porque toda la familia mantiene un hilo conductor: **los parámetros de variación son ciudadanos de primera clase del proceso evolutivo**.


### 1.3 El cromosoma extendido: la diferencia estructural fundamental

Para visualizar concretamente la diferencia entre el AGS y una EE, comparemos cómo representa cada uno a un individuo en un problema de optimización de dimensión $n = 4$.

**En el AGS** (codificación binaria con 8 bits por variable):

```
individuo = [ 0 1 1 0 1 0 0 1 | 1 0 0 1 1 1 0 0 | 0 0 1 1 0 1 1 0 | 1 1 0 0 0 1 0 1 ]
               ←─── x₁ ────→   ←─── x₂ ────→   ←─── x₃ ────→   ←─── x₄ ────→
               (32 bits en total, sin información sobre cómo mutar)
```

**En una EE** (codificación real con autoadaptación):

```
individuo = [ x₁,   x₂,   x₃,   x₄   |  σ₁,   σ₂,   σ₃,   σ₄  ]
              2.31  -0.87   1.54   0.23    0.41   0.12   0.38   0.09
              ←────── variables ──────→   ←── parámetros de estrategia ──→
              (las variables del problema)  (instrucciones para mutar cada variable)
```

Cuando este individuo se reproduce, genera un hijo aplicando:

$$\sigma_i' = \sigma_i \cdot \exp(\tau' \cdot \mathcal{N}(0,1) + \tau \cdot \mathcal{N}_i(0,1))$$

$$x_i' = x_i + \sigma_i' \cdot \mathcal{N}_i(0,1)$$

Las $\sigma_i$ mutan primero, y luego guían la mutación de las $x_i$. Si el hijo es bueno, sus $\sigma_i'$ —que produjeron esa buena solución— sobreviven. Si es malo, esas $\sigma_i'$ desaparecen. **Las instrucciones de mutación que funcionan se heredan**.

La siguiente celda construye una visualización directa de esta diferencia:

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# VISUALIZACIÓN COMPARATIVA: CROMOSOMA AGS vs. EE
# ════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle('Representación cromosómica: AGS vs. Estrategia Evolutiva', 
             fontsize=14, fontweight='bold', y=1.01)

# ─── Panel 1: Cromosoma AGS ───────────────────────────────────────────
ax = axes[0, 0]
n_bits = 32  # 8 bits × 4 variables
bits = rng.integers(0, 2, n_bits)
colores_bits = ['#24398A' if b == 1 else '#e0e0e0' for b in bits]

for i, (bit, color) in enumerate(zip(bits, colores_bits)):
    rect = plt.Rectangle([i, 0], 0.9, 1, facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i + 0.45, 0.5, str(bit), ha='center', va='center', fontsize=8,
            color='white' if bit == 1 else '#555', fontweight='bold')

# Separadores entre variables
for sep in [8, 16, 24]:
    ax.axvline(sep, color='#EBA93B', linewidth=3, zorder=5)

# Etiquetas de variables
for i, var in enumerate(['x₁', 'x₂', 'x₃', 'x₄']):
    ax.text(i*8 + 4, -0.5, var, ha='center', va='top', fontsize=12,
            color='#24398A', fontweight='bold')

ax.set_xlim(-0.5, n_bits + 0.5)
ax.set_ylim(-0.9, 1.5)
ax.set_title('AGS: Cromosoma binario', fontsize=12, fontweight='bold', color='#24398A')
ax.axis('off')
ax.text(16, 1.35, '32 bits — sin información sobre intensidad de mutación',
        ha='center', fontsize=9, color='#666', style='italic')

# ─── Panel 2: Cromosoma EE ────────────────────────────────────────────
ax = axes[0, 1]
n = 4
x_vals  = rng.uniform(-2, 2, n)
s_vals  = rng.uniform(0.05, 0.6, n)

# Variables x
cmap_x = plt.cm.Blues
for i, xv in enumerate(x_vals):
    norm_val = (xv + 2) / 4  # normalizar a [0,1]
    rect = plt.Rectangle([i*1.2, 1.2], 1.0, 0.8, 
                          facecolor=cmap_x(0.3 + 0.6*norm_val), 
                          edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(i*1.2 + 0.5, 1.6, f'{xv:.2f}', ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(i*1.2 + 0.5, 1.05, f'x{i+1}', ha='center', va='top',
            fontsize=10, color='#24398A')

# Parámetros σ
cmap_s = plt.cm.Oranges
for i, sv in enumerate(s_vals):
    norm_val = sv / 0.6
    rect = plt.Rectangle([i*1.2, 0.0], 1.0, 0.8, 
                          facecolor=cmap_s(0.3 + 0.6*norm_val), 
                          edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(i*1.2 + 0.5, 0.4, f'{sv:.2f}', ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(i*1.2 + 0.5, -0.15, f'σ{i+1}', ha='center', va='top',
            fontsize=10, color='#e65100')

# Etiquetas de sección
ax.text(-0.5, 1.6, 'Variables', fontsize=10, color='#24398A', fontweight='bold', va='center')
ax.text(-0.5, 0.4, 'Estrategia', fontsize=10, color='#e65100', fontweight='bold', va='center')

# Separador
ax.axhline(0.9, color='#666', linestyle='--', linewidth=1.5, alpha=0.5)

ax.set_xlim(-1.2, n*1.2 + 0.2)
ax.set_ylim(-0.5, 2.4)
ax.set_title('EE: Cromosoma extendido (variables + estrategia)', fontsize=12, 
             fontweight='bold', color='#24398A')
ax.axis('off')
ax.text(n*0.6, 2.3, '2n valores reales — σᵢ evoluciona junto con xᵢ',
        ha='center', fontsize=9, color='#666', style='italic')

# ─── Panel 3: σ a lo largo de generaciones — AGS ──────────────────────
ax = axes[1, 0]
gens = np.arange(100)
ax.axhline(0.3, color='#F44336', linewidth=2.5, label='σ = 0.3 (fija)')
ax.fill_between(gens, 0.28, 0.32, alpha=0.2, color='#F44336')

# Zona óptima hipotética
sigma_opt = 0.8 * np.exp(-gens / 30) + 0.03
ax.plot(gens, sigma_opt, color='#4CAF50', linewidth=2, linestyle='--',
        label='σ ideal (varía con el paisaje)')

ax.set_xlabel('Generación'); ax.set_ylabel('σ')
ax.set_title('AGS: σ no se adapta al paisaje', fontsize=11, color='#24398A')
ax.legend(fontsize=9); ax.grid(True, alpha=0.4)
ax.set_ylim(0, 1.0)
ax.annotate('σ ideal decrece\ncon la convergencia', xy=(50, 0.15), fontsize=9, color='#4CAF50',
            xytext=(60, 0.45), arrowprops=dict(arrowstyle='->', color='#4CAF50'))

# ─── Panel 4: σ adaptativa en EE ─────────────────────────────────────
ax = axes[1, 1]

# Simular evolución de σ en una EE
tau = 1 / np.sqrt(2 * np.sqrt(4))  # parámetro lognormal para n=4
tau_prime = 1 / np.sqrt(2 * 4)
sigma_ee = np.zeros((4, 100))
sigma_ee[:, 0] = [0.5, 0.5, 0.5, 0.5]
rng_local = np.random.default_rng(7)

for t in range(1, 100):
    global_noise = rng_local.normal(0, 1)
    for i in range(4):
        local_noise = rng_local.normal(0, 1)
        sigma_ee[i, t] = max(sigma_ee[i, t-1] * np.exp(tau_prime*global_noise + tau*local_noise), 1e-4)
        # Sesgo artificial hacia convergencia para ilustración
        if sigma_ee[i, t] > sigma_ee[i, t-1] and t > 20:
            sigma_ee[i, t] *= 0.92

colors_ee = ['#2196F3', '#E91E63', '#FF9800', '#4CAF50']
for i in range(4):
    ax.plot(sigma_ee[i], color=colors_ee[i], linewidth=1.5, alpha=0.8, label=f'σ{i+1}')

ax.set_xlabel('Generación'); ax.set_ylabel('σ')
ax.set_title('EE: σ evoluciona autónomamente', fontsize=11, color='#24398A')
ax.legend(fontsize=9, loc='upper right'); ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

### Qué observar en esta figura

Los cuatro paneles sintetizan la diferencia estructural entre los dos enfoques.

En la fila superior se contrasta la arquitectura del cromosoma. El AGS almacena únicamente bits que codifican los valores de las variables; no hay espacio en el cromosoma para información sobre cómo mutar. La EE, en cambio, lleva una segunda capa que contiene los parámetros de estrategia: cada variable tiene su propio σ asociado, y ambas capas evolucionan juntas.

En la fila inferior se ve la consecuencia dinámica de esta diferencia. El AGS mantiene σ constante a lo largo de todas las generaciones, independientemente de si el algoritmo está explorando zonas nuevas o afinando una solución casi convergida. La curva punteada verde marca la intensidad de mutación que sería óptima en cada fase: alta al principio, baja al final. La línea roja del AGS nunca coincide con ella.

Las trayectorias de σ en la EE muestran el comportamiento opuesto: cada componente del vector de estrategia sigue su propia trayectoria, respondiendo a la geometría local del paisaje en esa dirección. La convergencia hacia valores pequeños no es programada — emerge del propio proceso evolutivo.

<div style="background: linear-gradient(135deg, #e3f2fd, #bbdefb); border-left: 5px solid #1565c0; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>Reflexión:</strong> Nótese que los σ de cada dimensión convergen a valores distintos. Esto no es un defecto — es precisamente la señal de que el algoritmo ha aprendido que diferentes direcciones del espacio requieren diferentes intensidades de exploración. Esta información es exactamente lo que un optimizador basado en gradiente obtiene de la matriz Hessiana, pero aquí se infiere sin calcular derivadas.
</div>

### 1.4 Resumen: la taxonomía de las diferencias

Antes de profundizar en los mecanismos de las EE, conviene tener clara la taxonomía completa de lo que cambia. No se trata solo de una diferencia técnica en la representación: es una diferencia filosófica sobre el rol de los parámetros del algoritmo.

| Categoría | AGS | Estrategia Evolutiva | Implicación práctica |
|---|---|---|---|
| **¿Qué evoluciona?** | Solo las variables del problema | Variables + parámetros de variación | Las EE hacen metaoptimización implícita |
| **Tipo de espacio** | Discreto / binario | Continuo, ℝⁿ | Las EE no necesitan codificación |
| **Operador principal** | Cruza (intercambio de bloques) | Mutación gaussiana (perturbación vectorial) | Las EE explotan la geometría euclídea |
| **Selección** | Probabilística (torneo, ruleta) | Determinista (mejores μ de λ) | Las EE tienen presión selectiva controlada y predecible |
| **Parámetros del usuario** | Pop, n_gen, σ, p_cruce, p_mut | Pop, n_gen; σ₀ opcional | Las EE requieren menos ajuste manual |
| **Escalado** | Pierde eficiencia en alta dimensión | Diseñadas para alta dimensión | Las EE son la opción natural en n > 10 continuo |

<br>

En las siguientes secciones construiremos los mecanismos de las EE desde cero: primero la geometría de la mutación gaussiana, luego la autoadaptación lognormal, y finalmente CMA-ES como culminación de esta línea de ideas.

---

## 2. Representación y notación (μ, λ)

Las Estrategias Evolutivas tienen una notación precisa que comunica en dos números cómo opera el algoritmo. Entender esta notación es entender la lógica de selección de las EE.

### 2.1 Los dos números que definen una EE

**μ (mu)** — número de padres que sobreviven y se reproducen.  
**λ (lambda)** — número de hijos que se generan en cada generación.

Con estos dos números hay dos modos de operación posibles, y la diferencia entre ellos es más profunda de lo que parece:

| Notación | Selección | Elitismo | Cuándo usar |
|---|---|---|---|
| **(μ + λ)-ES** | Los mejores μ de **padres ∪ hijos** | ✅ Sí — los padres compiten con sus hijos | Paisajes suaves; cuando perder el mejor es inaceptable |
| **(μ, λ)-ES** | Los mejores μ de **solo los hijos** | ❌ No — los padres siempre desaparecen | Paisajes rugosos, ruidosos o no estacionarios |

Restricción estructural: en **(μ, λ)** se requiere que **λ ≥ μ** — deben nacer suficientes hijos para seleccionar μ de entre ellos.

<div style="background: linear-gradient(135deg, #f3e5f5, #e1bee7); border-left: 5px solid #6a1b9a; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>💡 Analogía:</strong> (μ+λ) es una empresa donde los veteranos compiten con los recién contratados y los mejores de ambos grupos conservan su puesto. (μ,λ) es una empresa donde cada generación de empleados se jubila por completo al nacer la siguiente — solo los nuevos compiten entre sí. El segundo esquema parece más duro, pero fuerza al algoritmo a explorar activamente en lugar de aferrarse a soluciones pasadas.
</div>

### 2.2 Flujo de una generación

El ciclo completo de una generación en una EE es:

```
PADRES (μ individuos)
    │
    ├─ Recombinación intermedia (opcional): mezclar padres para crear λ padres "virtuales"
    │
    ├─ Mutación lognormal de σ:   σᵢ' = σᵢ · exp(τ'·N(0,1) + τ·Nᵢ(0,1))
    │
    └─ Mutación gaussiana de x:   xᵢ' = xᵢ + σᵢ'·Nᵢ(0,1)
                                    ↓
                               HIJOS (λ individuos)
                                    │
                    ┌───────────────┴───────────────┐
                    │                               │
               (μ+λ)-ES                        (μ,λ)-ES
           Mejores μ de                     Mejores μ de
           padres ∪ hijos                   solo los hijos
                    │                               │
                    └───────────────┬───────────────┘
                                    │
                         NUEVA GENERACIÓN (μ padres)
```

### 2.3 Ratios típicos de μ y λ

La elección de μ y λ no es arbitraria. Hay una relación empírica bien establecida:  
el **ratio λ/μ ≈ 7** es un punto de partida robusto para muchos problemas.

La intuición es directa: si generamos pocos hijos (λ/μ pequeño), cada hijo tiene alta probabilidad de sobrevivir y la presión selectiva es baja — poca dirección en la búsqueda. Si generamos muchos hijos (λ/μ grande), solo el mejor 1/7 sobrevive — alta presión selectiva, búsqueda más dirigida pero más costosa computacionalmente.

Configuraciones comunes en la literatura:

| Configuración | μ | λ | λ/μ | Uso típico |
|---|---|---|---|---|
| Minimalista | 1 | 1 | 1.0 | (1+1)-ES: hill climbing estocástico |
| Clásica | 5 | 35 | 7.0 | Problemas de baja dimensión |
| Estándar | 15 | 100 | 6.7 | Benchmark general |
| Alta dimensión | 30 | 200 | 6.7 | n > 50 variables |
| CMA-ES default | ~7 | ~14 | 2.0 | Calculado automáticamente de n |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# VISUALIZACIÓN COMPARATIVA: (μ+λ)-ES vs. (μ,λ)-ES
#
# Objetivo: mostrar visualmente la diferencia en presión selectiva
# y comportamiento de convergencia entre los dos esquemas.
# Función de prueba: Rosenbrock 2D (conocida de la sección anterior)
# ════════════════════════════════════════════════════════════════════════

def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100*(y - x**2)**2


def ee_step(padres, sigma_padres, mu, lam, modo='plus', rng_local=None):
    """
    Una generación de EE con autoadaptación lognormal isótropa (σ escalar).
    modo: 'plus'  → (μ+λ): selección de padres ∪ hijos
          'comma' → (μ,λ): selección solo de hijos
    """
    n = padres.shape[1]
    tau = 1 / np.sqrt(2 * n)  # parámetro lognormal global

    # ── Generar λ hijos ──────────────────────────────────────────────
    hijos = np.zeros((lam, n))
    sigma_hijos = np.zeros(lam)

    for k in range(lam):
        # Seleccionar padre al azar (recombinación uniforme implícita)
        idx = rng_local.integers(0, mu)
        padre = padres[idx]
        s = sigma_padres[idx]

        # Autoadaptación de σ
        s_new = max(s * np.exp(tau * rng_local.standard_normal()), 1e-6)
        sigma_hijos[k] = s_new

        # Mutación gaussiana
        hijos[k] = padre + s_new * rng_local.standard_normal(n)

    # ── Selección ─────────────────────────────────────────────────────
    if modo == 'plus':
        pool_x = np.vstack([padres, hijos])
        pool_s = np.concatenate([sigma_padres, sigma_hijos])
    else:  # comma
        pool_x = hijos
        pool_s = sigma_hijos

    fitness = np.array([rosenbrock(ind) for ind in pool_x])
    idx_sorted = np.argsort(fitness)[:mu]  # minimizamos

    return pool_x[idx_sorted], pool_s[idx_sorted], fitness[idx_sorted[0]]


def run_ee(mu, lam, modo, n_gen=150, seed=42):
    """Ejecuta una EE completa y devuelve historial de convergencia."""
    rng_local = np.random.default_rng(seed)
    padres = rng_local.uniform(-2, 2, (mu, 2))
    sigma  = np.full(mu, 0.5)
    hist_fitness = []
    hist_sigma   = []
    hist_best    = []

    for _ in range(n_gen):
        padres, sigma, best_f = ee_step(padres, sigma, mu, lam, modo, rng_local)
        hist_fitness.append(best_f)
        hist_sigma.append(sigma.mean())
        hist_best.append(padres[0].copy())

    return np.array(hist_fitness), np.array(hist_sigma), np.array(hist_best)


# ── Ejecutar los cuatro escenarios ────────────────────────────────────
MU, LAM = 5, 35  # ratio 7:1 clásico
N_GEN   = 150

f_plus,  s_plus,  traj_plus  = run_ee(MU, LAM, 'plus',  N_GEN)
f_comma, s_comma, traj_comma = run_ee(MU, LAM, 'comma', N_GEN)

print(f'(μ+λ)-ES: mejor f = {f_plus[-1]:.6f}  |  σ final = {s_plus[-1]:.5f}')
print(f'(μ,λ)-ES: mejor f = {f_comma[-1]:.6f}  |  σ final = {s_comma[-1]:.5f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FIGURA: Comparativa (μ+λ) vs. (μ,λ) — trayectorias y convergencia
# ════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35)

# Paisaje Rosenbrock
xx, yy = np.meshgrid(np.linspace(-2.2, 2.2, 300), np.linspace(-1.5, 3.5, 300))
zz_log = np.log1p((1-xx)**2 + 100*(yy - xx**2)**2)

# ─── Trayectoria (μ+λ) ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.contourf(xx, yy, zz_log, levels=25, cmap='YlOrRd', alpha=0.75)
ax1.contour(xx, yy, zz_log, levels=8, colors='white', alpha=0.25, linewidths=0.5)
ax1.plot(traj_plus[:, 0], traj_plus[:, 1], '-o',
         color='#1565c0', markersize=2.5, linewidth=1.5, alpha=0.8)
ax1.plot(traj_plus[0,0], traj_plus[0,1], 's', color='cyan', markersize=10, zorder=6, label='Inicio')
ax1.plot(1, 1, '*', color='gold', markersize=14, zorder=7, label='Óptimo (1,1)')
ax1.set_title(f'(μ+λ)-ES  con μ={MU}, λ={LAM}', fontsize=11, fontweight='bold', color='#1565c0')
ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂')
ax1.legend(fontsize=8, loc='upper left')
ax1.set_xlim(-2.2, 2.2); ax1.set_ylim(-1.5, 3.5)
ax1.annotate(f'f* = {f_plus[-1]:.4f}', xy=(0.97, 0.04),
             xycoords='axes fraction', ha='right', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='#1565c0', alpha=0.25))

# ─── Trayectoria (μ,λ) ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.contourf(xx, yy, zz_log, levels=25, cmap='YlOrRd', alpha=0.75)
ax2.contour(xx, yy, zz_log, levels=8, colors='white', alpha=0.25, linewidths=0.5)
ax2.plot(traj_comma[:, 0], traj_comma[:, 1], '-o',
         color='#c62828', markersize=2.5, linewidth=1.5, alpha=0.8)
ax2.plot(traj_comma[0,0], traj_comma[0,1], 's', color='cyan', markersize=10, zorder=6, label='Inicio')
ax2.plot(1, 1, '*', color='gold', markersize=14, zorder=7, label='Óptimo (1,1)')
ax2.set_title(f'(μ,λ)-ES  con μ={MU}, λ={LAM}', fontsize=11, fontweight='bold', color='#c62828')
ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂')
ax2.legend(fontsize=8, loc='upper left')
ax2.set_xlim(-2.2, 2.2); ax2.set_ylim(-1.5, 3.5)
ax2.annotate(f'f* = {f_comma[-1]:.4f}', xy=(0.97, 0.04),
             xycoords='axes fraction', ha='right', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='#c62828', alpha=0.25))

# ─── Esquema estructural de selección ────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')
ax3.set_xlim(0, 10); ax3.set_ylim(0, 10)

ax3.text(5, 9.5, 'Esquema de selección', ha='center', fontsize=12,
         fontweight='bold', color='#24398A')

# (μ+λ)
ax3.text(2.5, 8.5, '(μ+λ)-ES', ha='center', fontsize=11, color='#1565c0', fontweight='bold')
for i in range(5):
    y = 7.3 - i*0.55
    color = '#1565c0' if i < MU else '#90a4ae'
    label = f'P{i+1}' if i < MU else f'H{i-MU+1}'
    ax3.add_patch(plt.Rectangle([0.3, y], 1.8, 0.45, facecolor=color, alpha=0.7, edgecolor='white'))
    ax3.text(1.2, y+0.22, label, ha='center', va='center', color='white', fontsize=9, fontweight='bold')
# hijos
for i in range(5):
    y = 7.3 - (i+5)*0.55
    ax3.add_patch(plt.Rectangle([0.3, y], 1.8, 0.45, facecolor='#EBA93B', alpha=0.7, edgecolor='white'))
    ax3.text(1.2, y+0.22, f'H{i+1}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
ax3.text(1.2, 1.3, '⬆ μ mejores\nde todo el pool', ha='center', fontsize=9, color='#1565c0')
ax3.add_patch(plt.FancyArrowPatch((1.2, 4.4), (1.2, 1.75),
                                   arrowstyle='->', color='#1565c0', lw=2,
                                   mutation_scale=18))

# (μ,λ)
ax3.text(7.5, 8.5, '(μ,λ)-ES', ha='center', fontsize=11, color='#c62828', fontweight='bold')
# padres (tachados)
for i in range(5):
    y = 7.3 - i*0.55
    ax3.add_patch(plt.Rectangle([5.5, y], 1.8, 0.45, facecolor='#90a4ae', alpha=0.5, edgecolor='white'))
    ax3.text(6.4, y+0.22, f'P{i+1} ✗', ha='center', va='center', color='#555', fontsize=9)
# hijos seleccionados
for i in range(5):
    y = 7.3 - (i+5)*0.55
    color = '#c62828' if i < MU else '#EBA93B'
    ax3.add_patch(plt.Rectangle([5.5, y], 1.8, 0.45, facecolor=color, alpha=0.75, edgecolor='white'))
    ax3.text(6.4, y+0.22, f'H{i+1}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
ax3.text(6.4, 1.3, '⬆ μ mejores\nsolo de hijos', ha='center', fontsize=9, color='#c62828')
ax3.add_patch(plt.FancyArrowPatch((6.4, 4.4), (6.4, 1.75),
                                   arrowstyle='->', color='#c62828', lw=2,
                                   mutation_scale=18))

# ─── Convergencia del fitness ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
gens = np.arange(1, N_GEN + 1)
ax4.semilogy(gens, f_plus + 1e-10,  color='#1565c0', linewidth=2.5, label=f'(μ+λ)-ES  f*={f_plus[-1]:.5f}')
ax4.semilogy(gens, f_comma + 1e-10, color='#c62828', linewidth=2.5, label=f'(μ,λ)-ES   f*={f_comma[-1]:.5f}')
ax4.axhline(0, color='gold', linestyle='--', linewidth=1.5, label='Óptimo global (f=0)')
ax4.set_xlabel('Generación'); ax4.set_ylabel('Mejor f(x) — escala log')
ax4.set_title(f'Convergencia comparada — μ={MU}, λ={LAM} — Rosenbrock 2D', fontweight='bold')
ax4.legend(fontsize=10); ax4.grid(True, alpha=0.35)

# ─── Evolución de σ promedio ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(gens, s_plus,  color='#1565c0', linewidth=2, label='(μ+λ)-ES')
ax5.plot(gens, s_comma, color='#c62828', linewidth=2, label='(μ,λ)-ES')
ax5.set_xlabel('Generación'); ax5.set_ylabel('σ promedio')
ax5.set_title('Adaptación de σ durante la búsqueda', fontweight='bold')
ax5.legend(fontsize=10); ax5.grid(True, alpha=0.35)

plt.suptitle(f'Comparativa (μ+λ)-ES vs. (μ,λ)-ES  |  Rosenbrock 2D  |  μ={MU}, λ={LAM}',
             fontsize=13, fontweight='bold', y=1.01)
plt.show()

### Qué observar en esta figura

Los dos mapas de trayectoria muestran rutas cualitativamente distintas hacia el mismo óptimo. La (μ+λ)-ES avanza de forma más conservadora: retiene los mejores padres, lo que produce una trayectoria menos errática pero que en ocasiones se queda orbitando cerca del óptimo sin afinar. La (μ,λ)-ES muestra trayectorias más exploratorias al principio — puede alejarse temporalmente del mejor punto conocido — pero esa capacidad de "olvidar" soluciones pasadas le permite escapar de regiones subóptimas.

La curva de convergencia lo confirma en escala logarítmica. Ambos esquemas reducen el valor de la función de forma sostenida, pero con perfiles diferentes: (μ+λ) tiende a converger más rápido en las primeras generaciones (gracias al elitismo), mientras que (μ,λ) puede alcanzar valores finales comparables con mayor robustez en paisajes donde el elitismo es perjudicial.

El panel de σ es el más instructivo: en ambos casos el parámetro de estrategia decrece de forma autónoma a medida que el algoritmo converge — nadie programó ese comportamiento. Emerge de la presión selectiva: las mutaciones pequeñas que afinen la solución son más ventajosas en las etapas finales, y los individuos que las llevan en su σ tienen mayor probabilidad de sobrevivir.

<div style="background: linear-gradient(135deg, #e8f5e9, #c8e6c9); border-left: 5px solid #2e7d32; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>Regla práctica:</strong> Usa (μ+λ) cuando el paisaje es suave y la pérdida de la mejor solución encontrada sea inaceptable (por ejemplo, diseño de ingeniería con evaluaciones costosas). Usa (μ,λ) cuando el paisaje sea ruidoso, cambiante, o cuando sospechas que puede haber convergencia prematura — el no-elitismo actúa como mecanismo de diversificación forzada.
</div>

### 2.4 Efecto del ratio λ/μ sobre la presión selectiva

El ratio λ/μ controla directamente cuánta presión selectiva ejerce el algoritmo: cuántos hijos compiten por el mismo número de plazas de supervivencia. Una exploración sistemática de este parámetro es más informativa que cualquier descripción verbal.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EFECTO DEL RATIO λ/μ SOBRE CONVERGENCIA Y DIVERSIDAD
# ════════════════════════════════════════════════════════════════════════

MU_FIXED = 5
ratios   = [1, 3, 7, 15, 30]  # λ/μ
colores  = ['#9C27B0', '#2196F3', '#4CAF50', '#FF9800', '#F44336']
N_GEN    = 120

resultados_ratio = {}
for ratio in ratios:
    lam = MU_FIXED * ratio
    # (μ,λ) requiere lam >= mu; (1,1) lo manejamos como caso especial
    if lam < MU_FIXED:
        continue
    f_hist, s_hist, _ = run_ee(MU_FIXED, lam, 'comma', N_GEN, seed=0)
    resultados_ratio[ratio] = (f_hist, s_hist, lam)

fig, (ax_f, ax_s) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Efecto del ratio λ/μ — (μ,λ)-ES sobre Rosenbrock 2D — μ={MU_FIXED} fijo',
             fontsize=13, fontweight='bold')

for (ratio, (f_hist, s_hist, lam)), color in zip(resultados_ratio.items(), colores):
    gens = np.arange(1, N_GEN + 1)
    ax_f.semilogy(gens, f_hist + 1e-10, color=color, linewidth=2,
                  label=f'λ/μ = {ratio}  (λ={lam})')
    ax_s.plot(gens, s_hist, color=color, linewidth=2,
              label=f'λ/μ = {ratio}  (λ={lam})')

ax_f.axhline(0, color='gold', linestyle='--', linewidth=1.5, alpha=0.8, label='Óptimo')
ax_f.set_xlabel('Generación'); ax_f.set_ylabel('Mejor f(x) — escala log')
ax_f.set_title('Convergencia del fitness', fontweight='bold')
ax_f.legend(fontsize=9); ax_f.grid(True, alpha=0.35)

ax_s.set_xlabel('Generación'); ax_s.set_ylabel('σ promedio')
ax_s.set_title('Evolución de σ promedio', fontweight='bold')
ax_s.legend(fontsize=9); ax_s.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

# Tabla resumen
print('\nResumen final:')
print(f'{"ratio λ/μ":>12} {"λ":>6} {"f* final":>14} {"σ final":>12} {"Eval. totales":>16}')
print('-' * 64)
for ratio, (f_hist, s_hist, lam) in resultados_ratio.items():
    evals = N_GEN * lam
    print(f'{ratio:>12} {lam:>6} {f_hist[-1]:>14.6f} {s_hist[-1]:>12.5f} {evals:>16}')

### Qué observar en esta figura

Las curvas de convergencia muestran un patrón claro: ratios bajos (λ/μ = 1 o 3) convergen rápido en las primeras generaciones pero se detienen antes — la presión selectiva es insuficiente para escapar de mínimos locales en el valle de Rosenbrock. Ratios altos (λ/μ = 15 o 30) invierten más evaluaciones por generación pero alcanzan valores finales menores y producen un σ que decrece más ordenadamente.

La tabla de resumen revela el trade-off real: el mejor valor de f se obtiene con λ/μ grande, pero el costo en evaluaciones de la función objetivo crece proporcionalmente. En problemas donde cada evaluación es costosa (simulaciones numéricas, experimentos físicos), este trade-off es la decisión de diseño más importante de la EE.

La **regla λ/μ ≈ 7** emerge como un equilibrio razonable: es el punto donde la mejora marginal al aumentar el ratio empieza a disminuir, sin incurrir en el costo computacional de ratios muy grandes.

---

<div style="background: #f8f9fa; padding: 18px; border-radius: 8px; border: 1px solid #dee2e6; margin: 20px 0;">
<h4 style="color: #24398A; margin-top: 0;">Conceptos clave de esta sección</h4>
<ul style="line-height: 2.0; color: #333;">
<li><strong>μ</strong> — número de padres que se reproducen; controla cuánta información se conserva entre generaciones.</li>
<li><strong>λ</strong> — número de hijos generados; controla la exploración por generación.</li>
<li><strong>(μ+λ)</strong> — elitista: los padres compiten con los hijos. Convergencia rápida, riesgo de estancamiento.</li>
<li><strong>(μ,λ)</strong> — no elitista: solo los hijos sobreviven. Más robusto ante paisajes ruidosos o cambiantes.</li>
<li><strong>λ/μ ≈ 7</strong> — ratio de presión selectiva recomendado como punto de partida.</li>
<li><strong>σ decrece autónomamente</strong> — la presión selectiva filtra de forma natural las estrategias de mutación adecuadas para cada fase de la búsqueda.</li>
</ul>
</div>

## 3. Mutación gaussiana y su geometría

La mutación es el operador central de las EE. A diferencia del flip de bits del AGS —que actúa discretamente sobre una representación binaria— la mutación gaussiana actúa directamente sobre el espacio continuo del problema y tiene una geometría precisa que vale la pena entender a fondo.

### 3.1 La operación básica

Dado un individuo $\mathbf{x} \in \mathbb{R}^n$, la mutación gaussiana isótropa genera un hijo:

$$\mathbf{x}' = \mathbf{x} + \sigma \cdot \mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

donde $\sigma$ es un escalar que controla el radio de la perturbación. Los hijos se distribuyen sobre una **esfera** centrada en el padre: la probabilidad de caer en cualquier dirección es idéntica.

Cuando cada dimensión tiene su propio $\sigma_i$ (**mutación anisótropa**):

$$x_i' = x_i + \sigma_i \cdot z_i, \quad z_i \sim \mathcal{N}(0, 1)$$

los hijos se distribuyen sobre un **elipsoide** alineado con los ejes coordenados. El algoritmo puede explorar más en algunas dimensiones que en otras.

La versión más general —que veremos en CMA-ES— usa una matriz de covarianza completa $\mathbf{C}$:

$$\mathbf{x}' = \mathbf{x} + \sigma \cdot \mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{C})$$

y el elipsoide puede rotar en cualquier dirección del espacio — ya no tiene por qué estar alineado con los ejes.

| Variante | Parámetros de variación | Forma de la nube de hijos | Parámetros libres |
|---|---|---|---|
| Isótropa | σ (escalar) | Esfera | 1 |
| Anisótropa | σ₁, …, σₙ (vector) | Elipsoide alineado con ejes | n |
| Covarianza completa (CMA-ES) | σ, **C** (matriz n×n) | Elipsoide rotado arbitrariamente | 1 + n(n+1)/2 |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GEOMETRÍA DE LA MUTACIÓN GAUSSIANA: ESFERA, ELIPSOIDE Y ROTADO
# Visualizamos las nubes de hijos generadas por cada variante
# desde el mismo padre, sobre el paisaje de Schwefel 2D.
# ════════════════════════════════════════════════════════════════════════

import matplotlib.patches as patches
from matplotlib.patches import Ellipse

def schwefel_2d(x, y):
    return 418.9829*2 - (x*np.sin(np.sqrt(np.abs(x))) + y*np.sin(np.sqrt(np.abs(y))))

# ── Paisaje ───────────────────────────────────────────────────────────
LIM = 500
xx, yy = np.meshgrid(np.linspace(-LIM, LIM, 300), np.linspace(-LIM, LIM, 300))
zz = schwefel_2d(xx, yy)

# Padre de ejemplo
padre = np.array([-200.0, 300.0])
N_HIJOS = 400
rng_local = np.random.default_rng(0)

# ── Generar hijos bajo cada variante ──────────────────────────────────
# 1. Isótropa: σ escalar
sigma_iso = 80.0
hijos_iso = padre + sigma_iso * rng_local.standard_normal((N_HIJOS, 2))

# 2. Anisótropa: σ diferente por dimensión
sigma_aniso = np.array([30.0, 120.0])
hijos_aniso = padre + sigma_aniso * rng_local.standard_normal((N_HIJOS, 2))

# 3. Covarianza completa: elipsoide rotado 45°
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
D = np.diag([120.0, 30.0])  # semiejes
L = R @ D  # descomposición de Cholesky simplificada
hijos_rot = padre + (L @ rng_local.standard_normal((2, N_HIJOS))).T

# ── Figura ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
titulos = ['Isótropa  (σ escalar)\nnube esférica',
           'Anisótropa  (σ₁ ≠ σ₂)\nelipsoide alineado con ejes',
           'Covarianza completa\nelipsoide rotado (CMA-ES)']
nubes   = [hijos_iso, hijos_aniso, hijos_rot]
colores = ['#1565c0', '#2e7d32', '#6a1b9a']

for ax, titulo, hijos, color in zip(axes, titulos, nubes, colores):
    # Paisaje
    ax.contourf(xx, yy, zz, levels=20, cmap='YlOrRd', alpha=0.65)
    ax.contour(xx, yy, zz, levels=8, colors='white', alpha=0.2, linewidths=0.5)

    # Nube de hijos
    ax.scatter(hijos[:, 0], hijos[:, 1], color=color, s=8, alpha=0.35, zorder=3)

    # Padre
    ax.plot(*padre, 'o', color='white', markersize=12, zorder=5,
            markeredgecolor=color, markeredgewidth=2.5)
    ax.plot(*padre, 'o', color=color, markersize=6, zorder=6)

    # Óptimo Schwefel
    ax.plot(420.97, 420.97, '*', color='gold', markersize=12, zorder=7, label='Óptimo')

    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM)
    ax.set_title(titulo, fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.legend(fontsize=8, loc='lower right')

    # Flecha de dirección preferente (solo en anisótropa y rotada)
    if titulo.startswith('Aniso'):
        for dx, dy, lbl in [(0, 120, 'σ₂=120'), (30, 0, 'σ₁=30')]:
            ax.annotate('', xy=(padre[0]+dx, padre[1]+dy),
                        xytext=padre,
                        arrowprops=dict(arrowstyle='->', color='cyan', lw=2.5))
            ax.text(padre[0]+dx*0.6, padre[1]+dy*0.6+15, lbl,
                    color='cyan', fontsize=9, fontweight='bold')
    elif titulo.startswith('Covar'):
        # Eje mayor rotado
        ax.annotate('', xy=(padre[0]+L[0,0], padre[1]+L[1,0]),
                    xytext=padre,
                    arrowprops=dict(arrowstyle='->', color='cyan', lw=2.5))
        ax.text(padre[0]+L[0,0]*0.5-30, padre[1]+L[1,0]*0.5+20,
                'eje mayor', color='cyan', fontsize=9, fontweight='bold')

fig.suptitle('Geometría de la mutación gaussiana — Schwefel 2D', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Qué observar en esta figura

Los tres paneles muestran el mismo padre (círculo blanco con borde de color) y el mismo paisaje de Schwefel, pero con nubes de hijos de geometría radicalmente distinta.

La nube **esférica** explora todas las direcciones con idéntica intensidad. Es útil cuando no sabemos nada de la geometría del paisaje, pero ineficiente si el paisaje tiene estructura direccional: si el valle se orienta a 45°, la mitad del "presupuesto" de exploración se desperdicia en direcciones perpendiculares.

La nube **elipsoidal alineada con los ejes** explora más en la dirección $x_2$ que en $x_1$. Si el paisaje tiene un valle más ancho en esa dirección, este σ anisótropo lo aprovecha. Pero si el valle está rotado respecto a los ejes coordenados, sigue siendo ineficiente.

La nube **rotada** es la más expresiva: el elipsoide puede orientarse en cualquier dirección del espacio, adaptándose a la geometría real del paisaje. Esto es exactamente lo que aprende CMA-ES de forma automática.

<div style="background: linear-gradient(135deg, #e3f2fd, #bbdefb); border-left: 5px solid #1565c0; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>Conexión con la sección anterior:</strong> Los σᵢ distintos que vimos evolucionar en la sección 2 son exactamente la versión anisótropa: el algoritmo aprendió que distintas dimensiones requieren distintas intensidades de exploración. CMA-ES extiende esto al caso rotado, aprendiendo también las correlaciones entre dimensiones.
</div>

### 3.2 La Regla de los 1/5 de Rechenberg

Antes de la autoadaptación lognormal (que veremos en la sección siguiente), Rechenberg propuso en 1973 un criterio empírico para adaptar σ manualmente: la **Regla de los 1/5**.

La observación es la siguiente: sobre la función esfera $f(\mathbf{x}) = \|\mathbf{x}\|^2$, la tasa de éxito óptima de las mutaciones —es decir, la fracción de hijos que mejoran al padre— es aproximadamente $1/5 = 0.2$. Si la tasa de éxito observada es:

- **Mayor que 1/5** → σ es demasiado pequeño (las mutaciones son tan conservadoras que casi siempre mejoran, pero avanzan poco)
- **Menor que 1/5** → σ es demasiado grande (las mutaciones son tan agresivas que casi siempre empeoran)
- **Igual a 1/5** → σ está bien calibrado

El mecanismo de ajuste es sencillo: cada $k$ generaciones se comprueba la tasa de éxito y se ajusta σ multiplicando por un factor $c$:

$$\sigma \leftarrow \begin{cases} \sigma / c & \text{si } p_s > 1/5 \quad (\text{ampliar}) \\ \sigma \cdot c & \text{si } p_s < 1/5 \quad (\text{reducir}) \\ \sigma & \text{si } p_s = 1/5 \end{cases}$$

donde $c \approx 0.817$ es el valor recomendado por Rechenberg.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DEMOSTRACIÓN DE LA REGLA DE LOS 1/5
# Comparamos (1+1)-ES con σ fija vs. (1+1)-ES con Regla de 1/5
# sobre la función de Schwefel 2D
# ════════════════════════════════════════════════════════════════════════

def schwefel(xy):
    return 418.9829*len(xy) - np.sum(xy * np.sin(np.sqrt(np.abs(xy))))


def es_1_1_sigma_fija(sigma0, n_gen=500, seed=0):
    """(1+1)-ES con σ fija."""
    rng_l = np.random.default_rng(seed)
    x = rng_l.uniform(-500, 500, 2)
    f = schwefel(x)
    sigma = sigma0
    hist_f, hist_s = [f], [sigma]

    for _ in range(n_gen):
        hijo = x + sigma * rng_l.standard_normal(2)
        hijo = np.clip(hijo, -500, 500)
        f_hijo = schwefel(hijo)
        if f_hijo <= f:
            x, f = hijo, f_hijo
        hist_f.append(f)
        hist_s.append(sigma)

    return np.array(hist_f), np.array(hist_s)


def es_1_1_regla_1_5(sigma0, c=0.817, k=10, n_gen=500, seed=0):
    """(1+1)-ES con Regla de 1/5 de Rechenberg."""
    rng_l = np.random.default_rng(seed)
    x = rng_l.uniform(-500, 500, 2)
    f = schwefel(x)
    sigma = sigma0
    hist_f, hist_s = [f], [sigma]
    exitos_ventana = []

    for gen in range(n_gen):
        hijo = x + sigma * rng_l.standard_normal(2)
        hijo = np.clip(hijo, -500, 500)
        f_hijo = schwefel(hijo)
        exito = f_hijo <= f
        exitos_ventana.append(int(exito))
        if exito:
            x, f = hijo, f_hijo

        # Ajuste de σ cada k generaciones
        if (gen + 1) % k == 0 and len(exitos_ventana) >= k:
            ps = np.mean(exitos_ventana[-k:])
            if ps > 1/5:
                sigma = sigma / c
            elif ps < 1/5:
                sigma = sigma * c
            sigma = np.clip(sigma, 1e-4, 1000)

        hist_f.append(f)
        hist_s.append(sigma)

    return np.array(hist_f), np.array(hist_s)


# ── Ejecutar experimentos ─────────────────────────────────────────────
SIGMA0 = 100.0
N_GEN  = 600

f_fija, s_fija     = es_1_1_sigma_fija(SIGMA0, N_GEN)
f_regla, s_regla   = es_1_1_regla_1_5(SIGMA0, n_gen=N_GEN)

SCHWEFEL_OPT = 0.0

print(f'σ fija  : f* = {f_fija[-1]:.4f}  |  σ final = {s_fija[-1]:.4f}')
print(f'Regla 1/5: f* = {f_regla[-1]:.4f}  |  σ final = {s_regla[-1]:.4f}')

# ── Figura ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Regla de los 1/5 de Rechenberg — (1+1)-ES sobre Schwefel 2D', 
             fontsize=13, fontweight='bold')
gens = np.arange(N_GEN + 1)

# Convergencia
ax = axes[0]
ax.semilogy(gens, f_fija + 1,   color='#F44336', linewidth=2, label=f'σ fija = {SIGMA0}')
ax.semilogy(gens, f_regla + 1,  color='#4CAF50', linewidth=2, label='Regla de 1/5')
ax.axhline(SCHWEFEL_OPT + 1, color='gold', linestyle='--', linewidth=1.5, label='Óptimo global')
ax.set_xlabel('Generación'); ax.set_ylabel('f(x) — escala log')
ax.set_title('Convergencia del fitness', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)

# Evolución de σ
ax = axes[1]
ax.plot(gens, s_fija,  color='#F44336', linewidth=2, label='σ fija')
ax.plot(gens, s_regla, color='#4CAF50', linewidth=2, label='Regla de 1/5')
ax.axhline(1/5 * SIGMA0, color='gold', linestyle=':', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Generación'); ax.set_ylabel('σ')
ax.set_title('Evolución de σ', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)

# Tasa de éxito a lo largo del tiempo (Regla 1/5)
ax = axes[2]
ventana = 30
exitos = []
rng_check = np.random.default_rng(0)
x_tmp = rng_check.uniform(-500, 500, 2)
f_tmp = schwefel(x_tmp)
s_tmp = SIGMA0
for _ in range(N_GEN):
    hijo = x_tmp + s_tmp * rng_check.standard_normal(2)
    hijo = np.clip(hijo, -500, 500)
    f_h = schwefel(hijo)
    ok = f_h <= f_tmp
    exitos.append(int(ok))
    if ok: x_tmp, f_tmp = hijo, f_h
    if (len(exitos)) % 10 == 0:
        ps = np.mean(exitos[-10:])
        s_tmp = s_tmp / 0.817 if ps > 0.2 else s_tmp * 0.817
        s_tmp = np.clip(s_tmp, 1e-4, 1000)

ps_rolling = np.convolve(exitos, np.ones(ventana)/ventana, mode='valid')
ax.plot(ps_rolling, color='#2196F3', linewidth=1.8, label=f'Tasa éxito (ventana {ventana})')
ax.axhline(1/5, color='#EBA93B', linestyle='--', linewidth=2, label='Umbral 1/5 = 0.20')
ax.fill_between(range(len(ps_rolling)), 1/5, ps_rolling,
                where=ps_rolling > 1/5, alpha=0.2, color='#4CAF50', label='σ demasiado pequeño')
ax.fill_between(range(len(ps_rolling)), ps_rolling, 1/5,
                where=ps_rolling < 1/5, alpha=0.2, color='#F44336', label='σ demasiado grande')
ax.set_xlabel('Generación'); ax.set_ylabel('Tasa de éxito')
ax.set_title('Diagnóstico: tasa de éxito vs. umbral 1/5', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

### Qué observar en esta figura

El panel izquierdo muestra que la Regla de 1/5 produce una convergencia sostenida donde σ fija se detiene. Esto es especialmente visible en escala logarítmica: la curva verde sigue descendiendo en generaciones tardías, mientras la roja se aplana.

El panel central revela el mecanismo: σ en la Regla de 1/5 no decrece monotónicamente, sino que oscila respondiendo al paisaje. En las primeras generaciones (exploración amplia) puede aumentar temporalmente; en las etapas finales converge a valores pequeños de afinamiento.

El panel derecho es el diagnóstico en tiempo real: la tasa de éxito observada oscila alrededor del umbral 0.20. Las zonas verdes indican generaciones donde σ estaba demasiado pequeño (muchos éxitos → se amplía); las zonas rojas indican generaciones donde σ era demasiado grande (pocos éxitos → se reduce). La regla mantiene al algoritmo balanceado entre exploración y explotación.

<div style="background: linear-gradient(135deg, #fff3e0, #ffe0b2); border-left: 5px solid #e65100; padding: 20px; margin: 20px 0; border-radius: 5px;">
<strong>⚠️ Limitación de la Regla de 1/5:</strong> Funciona bien para la función esfera y problemas de baja dimensión, pero es un heurístico basado en una sola función de referencia. No considera correlaciones entre dimensiones ni la heterogeneidad del paisaje. La autoadaptación lognormal (siguiente sección) y CMA-ES son soluciones más robustas que no asumen nada sobre la geometría del problema.
</div>

### 3.3 ¿Por qué 1/5? La intuición geométrica

El valor 1/5 no es arbitrario: se deriva de un análisis teórico sobre la función esfera. Pero hay una forma más intuitiva de entenderlo.

Imaginemos un individuo en el borde de una esfera de radio $r$ centrada en el óptimo. Si mutamos con σ muy pequeño, el hijo casi siempre cae en la misma esfera o más cerca del óptimo: tasa de éxito cercana a 1, pero el avance neto por generación es minúsculo. Si mutamos con σ muy grande, el hijo puede caer en cualquier parte del espacio: tasa de éxito cercana a 0, con mucho desperdicio de evaluaciones.

El progreso esperado por evaluación —la eficiencia del algoritmo— tiene un máximo en algún punto intermedio. Rechenberg demostró que ese máximo se alcanza cuando la tasa de éxito es aproximadamente 1/5 en la función esfera. La siguiente celda visualiza esta curva:

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CURVA DE PROGRESO ESPERADO vs. TASA DE ÉXITO
# Ilustra por qué la tasa óptima es ≈ 1/5 en la función esfera
# ════════════════════════════════════════════════════════════════════════

def tasa_exito_esfera(sigma, r, n=2, n_muestras=5000, seed=0):
    """Estima tasa de éxito y progreso esperado para (1+1)-ES en la esfera."""
    rng_l = np.random.default_rng(seed)
    x0 = np.zeros(n); x0[0] = r  # punto en la esfera
    f0 = np.dot(x0, x0)
    exitos = 0
    progreso_total = 0.0
    for _ in range(n_muestras):
        hijo = x0 + sigma * rng_l.standard_normal(n)
        f_h = np.dot(hijo, hijo)
        if f_h < f0:
            exitos += 1
            progreso_total += (f0 - f_h)
    ps = exitos / n_muestras
    prog = progreso_total / n_muestras
    return ps, prog

r_ref = 10.0
sigmas_test = np.logspace(-1, 2, 60)  # de 0.1 a 100
tasas, progresos = [], []

for s in sigmas_test:
    ps, prog = tasa_exito_esfera(s, r_ref)
    tasas.append(ps)
    progresos.append(prog)

tasas     = np.array(tasas)
progresos = np.array(progresos)

# Encontrar σ óptimo
idx_opt = np.argmax(progresos)
sigma_opt = sigmas_test[idx_opt]
tasa_opt  = tasas[idx_opt]

# ── Figura ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Eficiencia de la mutación gaussiana en función esfera — (1+1)-ES',
             fontsize=13, fontweight='bold')

# Tasa de éxito vs σ
ax1.semilogx(sigmas_test, tasas, color='#2196F3', linewidth=2.5)
ax1.axhline(1/5, color='#EBA93B', linestyle='--', linewidth=2, label='Umbral 1/5')
ax1.axvline(sigma_opt, color='#4CAF50', linestyle=':', linewidth=2, label=f'σ óptimo ≈ {sigma_opt:.1f}')
ax1.scatter([sigma_opt], [tasa_opt], color='#4CAF50', s=100, zorder=5)
ax1.set_xlabel('σ (escala log)'); ax1.set_ylabel('Tasa de éxito p_s')
ax1.set_title('Tasa de éxito en función de σ', fontweight='bold')
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.35)
ax1.set_ylim(0, 1)

# Progreso esperado vs σ
ax2.semilogx(sigmas_test, progresos, color='#9C27B0', linewidth=2.5)
ax2.axvline(sigma_opt, color='#4CAF50', linestyle=':', linewidth=2,
            label=f'σ óptimo ≈ {sigma_opt:.1f}  →  p_s ≈ {tasa_opt:.2f}')
ax2.scatter([sigma_opt], [progresos[idx_opt]], color='#4CAF50', s=100, zorder=5)

# Regiones
ax2.axvspan(sigmas_test[0], sigma_opt, alpha=0.07, color='#4CAF50', label='Sub-exploración (σ pequeño)')
ax2.axvspan(sigma_opt, sigmas_test[-1], alpha=0.07, color='#F44336', label='Sobre-exploración (σ grande)')

ax2.set_xlabel('σ (escala log)'); ax2.set_ylabel('Progreso esperado por evaluación')
ax2.set_title('Progreso esperado en función de σ', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

print(f'σ óptimo encontrado: {sigma_opt:.2f}  |  tasa de éxito en ese punto: {tasa_opt:.3f}  ≈ 1/5 = {1/5:.3f}')

### Qué observar en esta figura

El panel izquierdo muestra que la tasa de éxito es una función monotónicamente decreciente de σ: mutaciones pequeñas casi siempre mejoran (p_s → 1), mutaciones grandes casi nunca mejoran (p_s → 0). La línea amarilla marca el umbral 1/5.

El panel derecho es el más revelador: el **progreso esperado por evaluación** tiene un máximo claro. A la izquierda del pico (σ demasiado pequeño) el progreso crece porque las mutaciones se vuelven más arriesgadas pero más rentables. A la derecha del pico (σ demasiado grande) el progreso colapsa porque las mutaciones son demasiado ruidosas. El σ óptimo cae aproximadamente donde p_s ≈ 1/5, validando empíricamente la regla de Rechenberg.

---

<div style="background: #f8f9fa; padding: 18px; border-radius: 8px; border: 1px solid #dee2e6; margin: 20px 0;">
<h4 style="color: #24398A; margin-top: 0;">Conceptos clave de esta sección</h4>
<ul style="line-height: 2.0; color: #333;">
<li><strong>Mutación isótropa</strong> — σ escalar; nube esférica de hijos. Un solo parámetro que mutar.</li>
<li><strong>Mutación anisótropa</strong> — vector σᵢ; nube elipsoidal alineada con los ejes. Puede adaptarse a geometría diagonal.</li>
<li><strong>Covarianza completa (CMA-ES)</strong> — σ + matriz <strong>C</strong>; elipsoide rotado arbitrariamente. Máxima expresividad.</li>
<li><strong>Regla de los 1/5</strong> — heurístico de Rechenberg: si la tasa de éxito supera 1/5, ampliar σ; si la tasa cae por debajo, reducirlo.</li>
<li><strong>Progreso esperado</strong> — tiene un máximo en σ óptimo; la regla de 1/5 aproxima ese máximo sin calcularlo explícitamente.</li>
<li><strong>Limitación</strong> — la regla de 1/5 es exacta solo para la función esfera. La autoadaptación lognormal (siguiente sección) generaliza este principio de forma elegante y sin suposiciones sobre el paisaje.</li>
</ul>
</div>